# 🎙️ Clone-only voice (fast) — Colab

Lean notebook that does **only** the step that needs a GPU: turning a lesson's
**script text** into speech in a **cloned voice**.

It deliberately SKIPS the slow, unnecessary steps of the full dub:
- ❌ no Whisper (we already have the script text)
- ❌ no Demucs (we already have clean music stems)
- ❌ no video upload / transcription / separation

You upload **one reference voice clip**; it outputs cloned-voice audio for
every scene of the chosen lesson. Then Claude mixes it with the music stem and
muxes it onto the video locally (music never gets dropped).

**Runtime → Change runtime type → GPU → Save**, then run top to bottom.

In [ ]:
# 1) GPU check
!nvidia-smi -L || echo 'NO GPU: Runtime -> Change runtime type -> GPU, then re-run.'

In [ ]:
# 2) Get the project (public repo). Uses the course-video branch.
import os
USER, REPO, BRANCH = 'PrashantRMarathe', 'voice-replacer', 'add-course-video'
if os.path.isdir(REPO):
    !cd {REPO} && git fetch --quiet && git checkout {BRANCH} --quiet && git pull --quiet
else:
    !git clone -b {BRANCH} https://github.com/{USER}/{REPO}.git
%cd {REPO}
print('in', os.getcwd(), '| branch:', BRANCH)

In [ ]:
# 3) Install ONLY the voice engine (no whisper, no demucs) — much smaller/faster
!pip install -q coqui-tts 'transformers<5' ffmpeg-python soundfile num2words
!pip install -q f5-tts 2>/dev/null || echo 'NOTE: f5-tts optional, skipped.'
print('voice engine installed')

In [ ]:
# 4) Configure the voice engine
import os
os.environ['COQUI_TOS_AGREED'] = '1'   # needed for XTTS
import config
# F5 = softest/most natural for English; set to 'xtts' for other languages.
config.TTS_BACKEND = 'f5'
config.LANGUAGE = 'en'
print('backend:', config.TTS_BACKEND)

In [ ]:
# 5) Upload ONE reference voice clip (clean 15-30s of the voice to clone)
from google.colab import files
print('Select your REFERENCE VOICE file (mp3/wav)...')
reference_path = list(files.upload().keys())[0]
print('Reference:', reference_path)

In [ ]:
# 6) Clone the voice for every scene of the chosen lesson.
# Pick the lesson's scripts.json (one key per scene -> narration text).
import json, os, time
import tts

LESSON_SCRIPTS = 'course_video/src/lessons/zd/scripts.json'   # Zero Defects
OUT_DIR = 'clones'
os.makedirs(OUT_DIR, exist_ok=True)

scripts = json.load(open(LESSON_SCRIPTS, encoding='utf-8'))
backend = tts.get_backend()   # loads the voice model ONCE
print(f'Cloning {len(scripts)} scenes with backend:', backend.name)

t0 = time.time()
for i, (key, text) in enumerate(scripts.items(), 1):
    out = f'{OUT_DIR}/{key}.wav'
    if os.path.exists(out):
        print(f'  [{i}/{len(scripts)}] skip {key}'); continue
    backend.synth_to_file(text=text, speaker_wav=reference_path,
                          language=config.LANGUAGE, out_path=out)
    print(f'  [{i}/{len(scripts)}] {key}  ({time.time()-t0:.0f}s elapsed)')
print('\nAll scenes cloned ->', OUT_DIR)

In [ ]:
# 7) Zip + download. Send clones.zip to Claude -> he mixes with the music stem
#    and muxes onto the video (music stays, no caption clash).
!zip -r clones.zip clones >/dev/null && echo 'zipped clones.zip'
from google.colab import files
files.download('clones.zip')